# ResNet18 tuned pipeline (CBIS-DDSM)

Includes:
- Class-weighted loss
- Two-stage training (head, then full fine-tuning)
- ReduceLROnPlateau scheduler
- Early stopping
- Validation/Test metrics: ROC-AUC, recall (MALIGNANT), confusion matrix

In [1]:
from pathlib import Path
from collections import Counter
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    recall_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [2]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
TRAIN_CSV = ROOT / "src/data/processed/manifest_train.csv"
TEST_CSV = ROOT / "src/data/processed/manifest_test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

patients = train_df["patient_id"].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)

tr_df = train_df[train_df["patient_id"].isin(train_pat)].copy()
val_df = train_df[train_df["patient_id"].isin(val_pat)].copy()

print("Train:", tr_df.shape, tr_df["label_name"].value_counts().to_dict())
print("Val:", val_df.shape, val_df["label_name"].value_counts().to_dict())
print("Test:", test_df.shape, test_df["label_name"].value_counts().to_dict())

Train: (2315, 10) {'BENIGN': 1383, 'MALIGNANT': 932}
Val: (549, 10) {'BENIGN': 300, 'MALIGNANT': 249}
Test: (422, 10) {'BENIGN': 248, 'MALIGNANT': 174}


In [3]:
class MammographyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path_local"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label


IMG_SIZE = 224
BATCH_SIZE = 16

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.05, contrast=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = MammographyDataset(tr_df, transform=train_tfms)
val_ds = MammographyDataset(val_df, transform=eval_tfms)
test_ds = MammographyDataset(test_df, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [4]:
# Class weights from train split
counts = Counter(tr_df["label"].tolist())
w0 = len(tr_df) / (2 * counts[0])
w1 = len(tr_df) / (2 * counts[1])
class_weights = torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)
class_weights

tensor([0.8369, 1.2420], device='cuda:0')

In [5]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [6]:
def freeze_backbone(m):
    for name, p in m.named_parameters():
        p.requires_grad = name.startswith("fc")


def unfreeze_all(m):
    for p in m.parameters():
        p.requires_grad = True


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total, correct = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            probs = torch.softmax(logits, dim=1)[:, 1]
            pred = logits.argmax(1)

            total_loss += loss.item() * y.size(0)
            correct += (pred == y).sum().item()
            total += y.size(0)

            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.extend(probs.cpu().numpy().tolist())

    metrics = {
        "loss": total_loss / total,
        "acc": correct / total,
        "auc": roc_auc_score(y_true, y_prob),
        "recall_malignant": recall_score(y_true, y_pred, pos_label=1),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }
    return metrics

In [7]:
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 20
EARLY_STOP_PATIENCE = 5

best_metric = -1.0
epochs_no_improve = 0

best_path = ROOT / "reports/models/resnet18_tuned_best.pt"
best_path.parent.mkdir(parents=True, exist_ok=True)

history = []

# Phase 1: train only head
freeze_backbone(model)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val = evaluate(model, val_loader, criterion)
    scheduler.step(val["auc"])

    row = {
        "phase": 1,
        "epoch": epoch,
        "tr_loss": tr_loss,
        "tr_acc": tr_acc,
        "va_loss": val["loss"],
        "va_acc": val["acc"],
        "va_auc": val["auc"],
        "va_recall_mal": val["recall_malignant"],
    }
    history.append(row)
    print(f"P1 E{epoch:02d} | tr_acc={tr_acc:.4f} | va_acc={val['acc']:.4f} va_auc={val['auc']:.4f} va_rec_mal={val['recall_malignant']:.4f}")

    if val["auc"] > best_metric:
        best_metric = val["auc"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_path)
    else:
        epochs_no_improve += 1


# Phase 2: full fine-tuning
unfreeze_all(model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val = evaluate(model, val_loader, criterion)
    scheduler.step(val["auc"])

    row = {
        "phase": 2,
        "epoch": epoch,
        "tr_loss": tr_loss,
        "tr_acc": tr_acc,
        "va_loss": val["loss"],
        "va_acc": val["acc"],
        "va_auc": val["auc"],
        "va_recall_mal": val["recall_malignant"],
    }
    history.append(row)
    print(f"P2 E{epoch:02d} | tr_acc={tr_acc:.4f} | va_acc={val['acc']:.4f} va_auc={val['auc']:.4f} va_rec_mal={val['recall_malignant']:.4f}")

    if val["auc"] > best_metric:
        best_metric = val["auc"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_path)
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at phase 2 epoch {epoch} (patience={EARLY_STOP_PATIENCE}).")
        break

print("Best val AUC:", round(best_metric, 4))
print("Best model:", best_path)

P1 E01 | tr_acc=0.5430 | va_acc=0.5628 va_auc=0.6062 va_rec_mal=0.3454
P1 E02 | tr_acc=0.6121 | va_acc=0.5756 va_auc=0.6127 va_rec_mal=0.9197
P1 E03 | tr_acc=0.6410 | va_acc=0.5756 va_auc=0.6106 va_rec_mal=0.3976
P1 E04 | tr_acc=0.6406 | va_acc=0.5647 va_auc=0.6128 va_rec_mal=0.6426
P1 E05 | tr_acc=0.6527 | va_acc=0.5537 va_auc=0.5878 va_rec_mal=0.2851
P2 E01 | tr_acc=0.6613 | va_acc=0.5993 va_auc=0.6459 va_rec_mal=0.5261
P2 E02 | tr_acc=0.7028 | va_acc=0.6211 va_auc=0.6659 va_rec_mal=0.5542
P2 E03 | tr_acc=0.7102 | va_acc=0.6321 va_auc=0.6677 va_rec_mal=0.6627
P2 E04 | tr_acc=0.7339 | va_acc=0.6503 va_auc=0.6885 va_rec_mal=0.6386
P2 E05 | tr_acc=0.7400 | va_acc=0.6448 va_auc=0.6962 va_rec_mal=0.5422
P2 E06 | tr_acc=0.7413 | va_acc=0.6230 va_auc=0.6925 va_rec_mal=0.5100
P2 E07 | tr_acc=0.7724 | va_acc=0.6485 va_auc=0.6984 va_rec_mal=0.6265
P2 E08 | tr_acc=0.7797 | va_acc=0.6521 va_auc=0.6982 va_rec_mal=0.6787
P2 E09 | tr_acc=0.7940 | va_acc=0.6321 va_auc=0.6992 va_rec_mal=0.5341
P2 E10

In [8]:
hist_df = pd.DataFrame(history)
hist_df.tail(10)

,phase,epoch,tr_loss,tr_acc,va_loss,va_acc,va_auc,va_recall_mal
15,2,11,0.412962,0.810799,0.694803,0.652095,0.707062,0.654618
16,2,12,0.394415,0.821598,0.712852,0.632058,0.700114,0.554217
17,2,13,0.384864,0.826350,0.702162,0.624772,0.706151,0.622490
18,2,14,0.358101,0.844924,0.725617,0.621129,0.711573,0.510040
19,2,15,0.350579,0.845356,0.710659,0.628415,0.708092,0.590361
20,2,16,0.326244,0.857451,0.735563,0.633880,0.708963,0.590361
21,2,17,0.334616,0.860043,0.749181,0.648452,0.716834,0.554217
22,2,18,0.340482,0.848380,0.727702,0.641166,0.706566,0.654618
23,2,19,0.308798,0.870410,0.744775,0.626594,0.710515,0.538153
24,2,20,0.306288,0.873002,0.738743,0.642987,0.713461,0.602410


In [9]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test = evaluate(model, test_loader, criterion)

print(f"Test loss: {test['loss']:.4f}")
print(f"Test acc: {test['acc']:.4f}")
print(f"Test AUC: {test['auc']:.4f}")
print(f"Test recall malignant: {test['recall_malignant']:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(test["y_true"], test["y_pred"]))

print("\nClassification report:")
print(classification_report(test["y_true"], test["y_pred"], target_names=["BENIGN", "MALIGNANT"]))

Test loss: 0.7332
Test acc: 0.6422
Test AUC: 0.6989
Test recall malignant: 0.6149

Confusion matrix:
[[164  84]
 [ 67 107]]

Classification report:
              precision    recall  f1-score   support

      BENIGN       0.71      0.66      0.68       248
   MALIGNANT       0.56      0.61      0.59       174

    accuracy                           0.64       422
   macro avg       0.64      0.64      0.64       422
weighted avg       0.65      0.64      0.64       422

